## Grouped Subject-Aware Modeling and Uncertainty

This notebook reports final grouped evaluation results using the preferred compact setup selected in notebook 02.

Scope remains compact and interview-defensible:
1. grouped regression forecasting,
2. grouped current-activity classification,
3. grouped split conformal uncertainty for regression.

This notebook also surfaces the focused ablation findings (feature, target, and fill sensitivity) that motivated the preferred setup.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.grouped_evaluation import ALPHA, RANDOM_SEED, run_grouped_evaluation

PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table.parquet"
METRICS_DIR = REPO_ROOT / "artifacts" / "metrics"
FIGURES_DIR = REPO_ROOT / "artifacts" / "figures"
MODELS_DIR = REPO_ROOT / "artifacts" / "models"
PREFERRED_SETUP_PATH = METRICS_DIR / "grouped_cv_preferred_setup_summary.csv"

preferred_setup_df = pd.read_csv(PREFERRED_SETUP_PATH)
preferred_target_col = str(preferred_setup_df.iloc[0]["preferred_target_col"])

print(f"Repo root: {REPO_ROOT}")
print(f"Processed table exists: {PROCESSED_PATH.exists()}")
print(f"Preferred regression target: {preferred_target_col}")
print(f"Using random seed: {RANDOM_SEED}")
print(f"Conformal alpha: {ALPHA}")


In [ ]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f"Missing processed model table: {PROCESSED_PATH}")

model_df = pd.read_parquet(PROCESSED_PATH).copy()
model_df = model_df.sort_values(["subject_id", "timestamp_s"]).reset_index(drop=True)

required_columns = [
    "subject_id",
    "timestamp_s",
    "activity_target",
    "activity_label",
    "heart_rate_bpm",
    preferred_target_col,
]
missing_required = [column for column in required_columns if column not in model_df.columns]
if missing_required:
    raise ValueError(f"Processed table is missing required columns: {missing_required}")

duplicate_count = int(model_df.duplicated(subset=["subject_id", "timestamp_s"]).sum())
if duplicate_count > 0:
    raise ValueError(f"Found duplicate subject-second rows: {duplicate_count}")

print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")
print(f"Subjects: {sorted(model_df['subject_id'].unique().tolist())}")
print(f"Target columns: {[c for c in model_df.columns if c.startswith('hr_target_')]}")
display(model_df.head())


## Run Grouped LOSO Evaluation

This cell executes the full grouped workflow and writes all metric tables and figures used for model selection and error-breakdown reporting.


In [ ]:
results = run_grouped_evaluation(
    processed_path=PROCESSED_PATH,
    metrics_dir=METRICS_DIR,
    figures_dir=FIGURES_DIR,
    models_dir=MODELS_DIR,
    random_seed=RANDOM_SEED,
    alpha=ALPHA,
    regression_target_col=preferred_target_col,
)

print("Grouped evaluation complete for preferred setup.")
print("Saved grouped metrics and figures with grouped_cv_ file names.")
print("Returned result keys:", sorted(results.keys()))


In [ ]:
feature_ablation_df = pd.read_csv(METRICS_DIR / "grouped_cv_feature_ablation_summary.csv")
target_comparison_df = pd.read_csv(METRICS_DIR / "grouped_cv_target_comparison_summary.csv")
fill_sensitivity_df = pd.read_csv(METRICS_DIR / "grouped_cv_fill_sensitivity_summary.csv")

print("Feature ablation summary:")
display(feature_ablation_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Target comparison summary:")
display(target_comparison_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Fill sensitivity summary:")
display(fill_sensitivity_df.sort_values("best_mean_mae").reset_index(drop=True))


In [ ]:
regression_fold_df = results["regression_fold"].copy()
regression_summary_df = results["regression_summary"].copy()
classification_fold_df = results["classification_fold"].copy()
classification_summary_df = results["classification_summary"].copy()
classification_per_class_df = results["classification_per_class"].copy()

print("Regression fold-level metrics:")
display(regression_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Regression grouped CV summary (mean/std/min/max across folds):")
display(regression_summary_df.sort_values("rank"))

print("Classification fold-level metrics:")
display(classification_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Classification grouped CV summary (mean/std/min/max across folds):")
display(classification_summary_df.sort_values("rank"))

print("Selected classification model per-class performance:")
display(classification_per_class_df.sort_values("support", ascending=False).reset_index(drop=True))


## Final Model Selection and Breakdown Reporting

Selection is driven by grouped CV summaries only, not one validation subject.

Rules used:
- Regression: lowest mean MAE across LOSO folds, tie-break by MAE standard deviation then mean RMSE.
- Classification: highest mean macro F1 across LOSO folds, tie-break by macro F1 standard deviation then mean accuracy.


In [ ]:
selected_models_df = results["selected_models"].copy()

regression_by_subject_df = results["regression_by_subject"].copy()
regression_by_activity_df = results["regression_by_activity"].copy()
classification_by_subject_df = results["classification_by_subject"].copy()
classification_by_activity_df = results["classification_by_activity"].copy()

print("Selected-model summary table:")
display(selected_models_df)

print("Regression performance by subject:")
display(regression_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Regression performance by activity:")
display(regression_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))

print("Classification performance by subject:")
display(classification_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Classification performance by activity:")
display(classification_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))


## Leakage and Coverage Checks

This section confirms fold-wise subject isolation and summarizes grouped conformal interval quality.


In [ ]:
subject_count = int(model_df["subject_id"].nunique())

reg_subject_coverage = (
    regression_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)
cls_subject_coverage = (
    classification_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

reg_subjects_per_model = regression_fold_df.groupby("model")["test_subject_id"].nunique()
cls_subjects_per_model = classification_fold_df.groupby("model")["test_subject_id"].nunique()

if not (reg_subjects_per_model == subject_count).all():
    raise ValueError("Regression grouped CV did not cover each subject exactly once per model.")
if not (cls_subjects_per_model == subject_count).all():
    raise ValueError("Classification grouped CV did not cover each subject exactly once per model.")

print("Leakage guard checks passed.")
print(f"Unique subjects in modeling table: {subject_count}")
print("Regression subjects per model:")
display(reg_subjects_per_model.rename("unique_test_subjects"))
print("Classification subjects per model:")
display(cls_subjects_per_model.rename("unique_test_subjects"))


## Grouped Conformal Uncertainty

Conformal intervals are computed fold by fold with disjoint subject sets:
- proper-train subjects fit the selected regression model,
- one separate calibration subject sets the conformal margin,
- the held-out test subject receives interval predictions.

This preserves subject isolation during uncertainty evaluation.


In [ ]:
conformal_fold_df = results["conformal_fold"].copy()
conformal_summary_df = results["conformal_summary"].copy()
conformal_by_subject_df = results["conformal_by_subject"].copy()
conformal_by_activity_df = results["conformal_by_activity"].copy()

print("Conformal fold summary:")
display(conformal_fold_df.sort_values("fold").reset_index(drop=True))

print("Conformal aggregate summary:")
display(conformal_summary_df)

print("Conformal coverage by subject:")
display(conformal_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Conformal coverage by activity:")
display(conformal_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))


In [ ]:
grouped_metric_files = sorted(METRICS_DIR.glob("grouped_cv_*.csv"))
grouped_figure_files = sorted(FIGURES_DIR.glob("grouped_cv_*.png"))

print("Grouped metric artifacts:")
for path in grouped_metric_files:
    print(path.relative_to(REPO_ROOT))

print("\nGrouped figure artifacts:")
for path in grouped_figure_files:
    print(path.relative_to(REPO_ROOT))


## Notes on Compactness and Defensibility

This workflow keeps the modeling story compact while answering the key ablation questions.

Regression models compared per setup:
- persistence baseline (current heart rate),
- linear regression,
- histogram gradient boosting regressor.

Classification models compared:
- logistic regression,
- random forest classifier.

The final story now answers:
- which targeted feature changes helped most,
- whether target reformulation changed forecast quality,
- whether outcomes are mostly persistence-driven,
- whether the preferred setup is still simple enough to explain clearly.


In [ ]:
# Compact interview-oriented summary pulled from grouped evaluation and ablation outputs.
reg_choice = selected_models_df[selected_models_df["task"] == "regression"].iloc[0]
cls_choice = selected_models_df[selected_models_df["task"] == "classification"].iloc[0]
preferred_setup_df = pd.read_csv(METRICS_DIR / "grouped_cv_preferred_setup_summary.csv")

best_reg_row = regression_summary_df.sort_values("mean_mae").iloc[0]
persistence_reg_row = regression_summary_df[regression_summary_df["model"] == "persistence_current_hr"].iloc[0]
persistence_gap = persistence_reg_row["mean_mae"] - best_reg_row["mean_mae"]

print("Preferred setup:")
display(preferred_setup_df)

print("Final model choices from grouped CV:")
print(
    f"Regression: {reg_choice['selected_model']} | mean MAE={reg_choice['selected_mean_mae']:.3f} "
    f"(runner-up margin={reg_choice['winner_margin']:.3f})"
)
print(
    f"Classification: {cls_choice['selected_model']} | mean macro F1={cls_choice['selected_mean_macro_f1']:.3f} "
    f"(runner-up margin={cls_choice['winner_margin']:.3f})"
)

print("\nPersistence sensitivity check:")
print(f"Best regression model MAE: {best_reg_row['mean_mae']:.3f}")
print(f"Persistence baseline MAE: {persistence_reg_row['mean_mae']:.3f}")
print(f"MAE gain vs persistence: {persistence_gap:.3f}")
if persistence_gap < 0.25:
    print("Interpretation: forecast task looks strongly persistence-driven.")
elif persistence_gap < 0.75:
    print("Interpretation: forecast task is partly persistence-driven, but model features add clear value.")
else:
    print("Interpretation: upgraded setup captures meaningful signal beyond persistence.")

print("\nKey risk surfaces to discuss:")
print("- Highest-regression-error activities:")
display(
    regression_by_activity_df.sort_values("mae", ascending=False).head(5)[
        ["activity_label", "rows", "mae", "rmse", "r2"]
    ]
)
print("- Lowest-classification-F1 classes:")
display(
    classification_per_class_df.sort_values("f1", ascending=True).head(5)[
        ["activity_label", "support", "precision", "recall", "f1"]
    ]
)


## Completion Checklist

This notebook now reports the final preferred setup with compact, subject-aware evaluation:
- focused ablation outputs for feature, target, and fill-policy decisions,
- grouped LOSO model selection for regression and classification,
- fold-level and aggregate model comparison outputs,
- selected-model summary with explicit rules and target column,
- by-subject and by-activity breakdown tables,
- per-class classification performance,
- grouped conformal uncertainty reporting.

All non-obvious choices are documented in project docs and decision log.
